In [1]:
import matplotlib
matplotlib.use('Qt5Agg')  # Important for VSCode interactivity

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import PolygonSelector
from matplotlib.path import Path
from matplotlib.colors import LinearSegmentedColormap
import scanpy as sc

import numpy as np
import os
import plotly.io as pio
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import scipy.sparse as sp
from anndata import AnnData



In [2]:
class PolygonAnnotator:
    def __init__(self, image, cmap='hot'):
        self.image = image
        self.coords = None
        self.done = False

        self.fig, self.ax = plt.subplots()
        self.ax.imshow(image, cmap=cmap)
        self.ax.set_title("Click to draw polygon. Double-click to finish.")

        self.selector = PolygonSelector(
            self.ax,
            self.onselect,
            useblit=True,
            props=dict(color='cyan', linewidth=2),
            handle_props=dict(marker='o', markersize=5, mec='cyan', mfc='cyan')
        )

        self.fig.canvas.mpl_connect('close_event', self._on_close)
        plt.show(block=True)  # Blocks until the window is closed

    def _on_close(self, event):
        self.done = True

    def onselect(self, verts):
        self.coords = verts
        print(f"Polygon drawn with {len(verts)} points.")
        plt.close(self.fig)  # Automatically close the window

    def get_mask(self):
        if not self.coords:
            print("No polygon was drawn.")
            return None
        ny, nx = self.image.shape
        y, x = np.mgrid[:ny, :nx]
        points = np.vstack((x.flatten(), y.flatten())).T
        path = Path(self.coords)
        return path.contains_points(points).reshape((ny, nx))

In [3]:
def get_mz_heatmap_image(adata, target_mz, tol=0.1):
    mz_axis = adata.var_names.astype(float).values
    mz_diff = np.abs(mz_axis - target_mz)
    if np.min(mz_diff) > tol:
        raise ValueError(f"No m/z found within ±{tol} of {target_mz}")
    mz_index = np.argmin(mz_diff)
    matched_mz = mz_axis[mz_index]

    intensities = adata.X[:, mz_index].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_index]
    x = adata.obs["x"].values.astype(int)
    y = adata.obs["y"].values.astype(int)

    width = x.max() + 1
    height = y.max() + 1
    image = np.zeros((height, width))
    image[y, x] = intensities

    return image, matched_mz

In [4]:
# Load and visualize the m/z heatmap from AnnData
input_file = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a_peaks_0.00001_top1000_9w_3p_5points.h5ad"
adata = sc.read_h5ad(input_file)
print(f"Loaded AnnData from: {input_file}")

Loaded AnnData from: /Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a_peaks_0.00001_top1000_9w_3p_5points.h5ad


In [5]:
def sort_adata_by_mz(adata, mz_column="mzs"):
    def to_float_with_warning(index, label="var_names"):
        mzs = pd.to_numeric(index, errors="coerce")
        n_nans = mzs.isna().sum()
        if n_nans > 0:
            print(f"⚠️ Warning: {n_nans} entries in {label} could not be converted to float and were set to NaN.")
        return mzs
    adata.var[mz_column] = to_float_with_warning(adata.var_names)
    adata = adata[:, adata.var[mz_column].sort_values().index]
    return adata


In [6]:
adata = sort_adata_by_mz(adata, mz_column="mzs")
adata.var

,mzs
136.0150,136.0150
136.0608,136.0608
137.0251,137.0251
138.0287,138.0287
139.0359,139.0359
...,...
1520.1588,1520.1588
1521.1633,1521.1633
1522.1682,1522.1682
1542.1475,1542.1475


In [7]:
# ---- Run ----

target_mz = 788.5
image, matched_mz = get_mz_heatmap_image(adata, target_mz)
print(f"Target m/z: {target_mz} → Matched m/z: {matched_mz}")

custom_cmap = LinearSegmentedColormap.from_list("custom_scale", [
    (0.0, "#000000"),  # black
    (0.10, "#000080"),  # navy
    (0.15, "#0000FF"),  # blue
    (0.30, "#8000FF"),  # purple-ish
    (0.45, "#FF0000"),  # red
    (0.60, "#FF8000"),  # orange
    (0.75, "#FFFF00"),  # yellow
    (1.0, "#FFFFFF")   # white
])

# Use the annotator with your image and colormap
annotator = PolygonAnnotator(image, cmap=custom_cmap)
mask = annotator.get_mask()

if mask is not None:
    masked_image = np.where(mask, image, 0)
    plt.imshow(masked_image, cmap=custom_cmap)
    plt.title(f"Masked m/z = {matched_mz:.4f}")
    plt.show()
else:
    print("No mask created.")

Target m/z: 788.5 → Matched m/z: 788.5027
Polygon drawn with 36 points.


In [8]:
# Get spatial coordinates
x = adata.obs["x"].astype(int).values
y = adata.obs["y"].astype(int).values

# Mask value at each spot
adata.obs["tissue"] = mask[y, x]  # Boolean mask (True = inside polygon)
adata.obs["tissue"] = adata.obs["tissue"].astype(int)  # 1 = inside, 0 = outside
print(adata.obs["tissue"].value_counts())

tissue
0    9441
1    7164
Name: count, dtype: int64


/var/folders/0n/w0dydx896p7b5chql0_cv6hr0000gn/T/ipykernel_81766/1596079847.py:6: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs["tissue"] = mask[y, x]  # Boolean mask (True = inside polygon)


In [9]:
adata1 = adata
adata.obs["tissue"]

0        0
1        0
2        0
3        0
4        0
        ..
16600    0
16601    0
16602    0
16603    0
16604    0
Name: tissue, Length: 16605, dtype: int64

In [10]:
adata = adata1
adata

AnnData object with n_obs × n_vars = 16605 × 1000
    obs: 'x', 'y', 'TIC', 'sample', 'batch', 'age_group', 'disease_status', 'tissue'
    var: 'mzs'

In [11]:
adata.var_names

Index(['136.0150', '136.0608', '137.0251', '138.0287', '139.0359', '154.0255',
       '155.0350', '156.0424', '159.0070', '160.0107',
       ...
       '1249.0300', '1255.0399', '1271.0072', '1494.1565', '1495.1523',
       '1520.1588', '1521.1633', '1522.1682', '1542.1475', '1544.1542'],
      dtype='object', length=1000)

In [12]:
def plot_mz_across_sample_custom_tic(adata, target_mz, tol=0.1):
    """
    Plots the TIC-normalized intensity of a specific m/z across all pixels using spatial coordinates.

    Parameters:
    - adata: AnnData object with MSI data
    - target_mz: float, the m/z value of interest
    - tol: float, tolerance for matching m/z (default ±0.1)
    """
    # Convert var_names to float (m/z axis)
    mz_axis = adata.var_names.astype(float).values

    # Find index of the m/z closest to the target within tolerance
    mz_diff = np.abs(mz_axis - target_mz)
    if np.min(mz_diff) > tol:
        raise ValueError(f"No m/z found within ±{tol} of {target_mz}")
    mz_index = np.argmin(mz_diff)
    matched_mz = mz_axis[mz_index]

    # Extract intensities for that m/z from all pixels
    intensities = adata.X[:, mz_index].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_index]

    # Normalize by TIC
    if "TIC" not in adata.obs.columns:
        raise KeyError("TIC not found in adata.obs. Compute TIC before using this function.")
    tic = adata.obs["TIC"].values
    norm_intensities = intensities / tic
    norm_intensities[np.isnan(norm_intensities)] = 0  # Avoid NaNs from division by 0

    # Extract spatial coordinates
    if "x" not in adata.obs.columns or "y" not in adata.obs.columns:
        raise KeyError("Spatial coordinates 'x' and 'y' not found in adata.obs")

    x = adata.obs["x"].values
    y = adata.obs["y"].values

    # Create DataFrame for plotting
    df = pd.DataFrame({"x": x, "y": y, "intensity": norm_intensities})

    # Plot as heatmap using plotly express
    fig = px.scatter(
        df,
        x="x",
        y="y",
        color="intensity",
        color_continuous_scale=[
            [0.0, "#000000"],  # black
            [0.10, "#000080"],  # navy
            [0.15, "#0000FF"],  # blue
            [0.30, "#8000FF"],  # purple-ish
            [0.45, "#FF0000"],  # red
            [0.60, "#FF8000"],  # orange
            [0.75, "#FFFF00"],  # yellow
            [1.0, "#FFFFFF"]   # white
        ],
        title=f"TIC-Normalized Intensity Map of m/z = {matched_mz:.4f}",
        labels={"intensity": "Normalized Intensity"},
        height=500
    )

    fig.update_layout(yaxis=dict(scaleanchor="x", autorange="reversed"))
    return fig

NameError: name 'foldchange' is not defined

In [14]:
def export_pixel_areas_to_csv_with_centroid(
    adata: AnnData,
    pixel_area: float = 100.0,
    output_csv: str = "pixel_area_summary.csv",
    x_key: str = "x",
    y_key: str = "y",
):
    """
    Calculates and exports per-m/z area, intensity sum, and centroid info to a CSV
    using all pixels without separating tissue and non-tissue regions.

    Parameters:
        adata : AnnData
            AnnData object with .X, .obs[x_key], .obs[y_key], and .var["mzs"] or .var_names
        pixel_area : float
            Area of a single pixel (e.g., in μm²)
        output_csv : str
            Output CSV filename
        x_key : str
            Key in adata.obs for x coordinates
        y_key : str
            Key in adata.obs for y coordinates
    """
    X = adata.X
    if sp.issparse(X):
        X = X.tocsr()

    if "mzs" in adata.var.columns:
        mz_values = adata.var["mzs"].values
    else:
        mz_values = pd.to_numeric(adata.var_names, errors="coerce")
        print("Warning: 'mzs' not found in var. Using var_names instead.")

    x_coords = adata.obs[x_key].values
    y_coords = adata.obs[y_key].values

    total_area = []
    total_sum = []
    centroid_x = []
    centroid_y = []

    for mz_idx in range(X.shape[1]):
        intensities = X[:, mz_idx].toarray().flatten() if sp.issparse(X) else X[:, mz_idx]

        # Area (number of non-zero pixels * pixel_area)
        total_area.append(np.sum(intensities > 0) * pixel_area)

        # Intensity sum
        total_sum.append(np.sum(intensities))

        # Centroid
        total = np.sum(intensities)
        if total > 0:
            centroid_x.append(np.sum(x_coords * intensities) / total)
            centroid_y.append(np.sum(y_coords * intensities) / total)
        else:
            centroid_x.append(np.nan)
            centroid_y.append(np.nan)

    df = pd.DataFrame({
        "m/z": mz_values,
        "total_area": total_area,
        "total_intensity_sum": total_sum,
        "centroid_x": centroid_x,
        "centroid_y": centroid_y,
    })

    df.to_csv(output_csv, index=False)
    print(f"Saved to {output_csv}")
    return df

In [15]:
df = export_pixel_areas_to_csv_with_centroid(
    adata,
    pixel_area = 1.0,
    output_csv = "all_data_pixel_area_summary.csv",
    x_key = "x",
    y_key = "y",
)

Saved to all_data_pixel_area_summary.csv


In [16]:
df

,m/z,total_area,total_intensity_sum,centroid_x,centroid_y
0,136.0150,15639.0,7371791.0,69.657867,73.930941
1,136.0608,15734.0,3734655.0,63.942228,58.469107
2,137.0251,16605.0,428154643.0,69.785415,83.693887
3,138.0287,16576.0,33900021.0,69.608645,81.122010
4,139.0359,16083.0,7500120.0,69.855193,78.372040
...,...,...,...,...,...
995,1520.1588,12521.0,888007.0,64.691163,55.186782
996,1521.1633,12396.0,792372.0,64.757237,55.634439
997,1522.1682,12509.0,859385.0,64.550489,55.576733
998,1542.1475,12391.0,694100.0,64.029550,53.913122


In [17]:
import matplotlib.pyplot as plt
from skimage import measure

# Get contours at a constant value of 0.5 (between 0 and 1 in binary masks)
contours = measure.find_contours(mask, level=0.5)

# Plot the image with lower opacity and overlay the contour lines
plt.imshow(image, cmap='gray', alpha=0.3)  # Faded background
for contour in contours:
    plt.plot(contour[:, 1], contour[:, 0], linewidth=2, color='red')  # (x, y) = (col, row)

plt.axis('off')
plt.title('Polygon Margin Lines')
plt.show()

In [18]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from skimage import measure

def plot_mz_centroids_with_mask_contours(
    df,
    image,
    mask=None,
    cmap="viridis",
    marker_size=100,
    figsize=(10, 8),
    show_margin=True,
    show_mask_contours=True,
    image_alpha=0.3,
    margin_linewidth=1.5,
    contour_color="red"
):
    """
    Plot centroid heatmap for each m/z using unified centroids.
    Optionally overlay the original image (faded) and polygon margin contours from mask.
    """
    required_cols = ["centroid_x", "centroid_y", "total_intensity_sum"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")

    norm_intensity = plt.Normalize(vmin=df["total_intensity_sum"].min(), vmax=df["total_intensity_sum"].max())
    cmap_obj = plt.colormaps[cmap]

    fig, ax = plt.subplots(figsize=figsize)

    # Show background image
    ax.imshow(image, cmap='gray', alpha=image_alpha)

    # Plot centroids
    for _, row in df.iterrows():
        if not np.isnan(row["centroid_x"]) and not np.isnan(row["centroid_y"]):
            color = cmap_obj(norm_intensity(row["total_intensity_sum"]))
            circle = patches.Circle(
                (row["centroid_x"], row["centroid_y"]),
                radius=marker_size / 2,
                linewidth=0.5,
                edgecolor="black",
                facecolor=color,
                alpha=0.8
            )
            ax.add_patch(circle)

    # Plot mask contours
    if show_mask_contours and mask is not None:
        contours = measure.find_contours(mask, level=0.5)
        for contour in contours:
            ax.plot(contour[:, 1], contour[:, 0], linewidth=margin_linewidth, color=contour_color)

    # Optional average centroid margin
    if show_margin:
        valid_centroids = df[["centroid_x", "centroid_y"]].dropna()
        if not valid_centroids.empty:
            avg_x = valid_centroids["centroid_x"].mean()
            avg_y = valid_centroids["centroid_y"].mean()
            ax.plot([avg_x - 100, avg_x + 100], [avg_y - 100, avg_y + 100],
                    color="black", linewidth=margin_linewidth, linestyle="--")

    # Colorbar
    sm = plt.cm.ScalarMappable(cmap=cmap_obj, norm=norm_intensity)
    sm.set_array([])
    plt.colorbar(sm, ax=ax, label="Total Intensity Sum")

    ax.set_aspect('equal')
    ax.set_xlabel("X Coordinate (μm)")
    ax.set_ylabel("Y Coordinate (μm)")
    ax.set_title("m/z Centroids with Unified Intensity")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [20]:
plot_mz_centroids_with_mask_contours(
    df,
    image,
    mask=None,
    cmap="viridis",
    marker_size=0.5,
    figsize=(6, 4),
    show_margin=False,
    show_mask_contours=True,
    image_alpha=0.3,
    margin_linewidth=1.5,
    contour_color="red"
)

In [21]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from skimage import measure

def plot_log_mz_centroids_with_mask_contours(
    df,
    image,
    mask=None,
    cmap="viridis",
    marker_size=100,
    figsize=(10, 8),
    show_margin=True,
    show_mask_contours=True,
    image_alpha=0.3,
    margin_linewidth=1.5,
    contour_color="red"
):
    """
    Plot centroid heatmap for each m/z using unified centroids with log-scaled intensity coloring.
    Optionally overlay the original image and mask contours.
    """
    required_cols = ["centroid_x", "centroid_y", "total_intensity_sum"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")

    # Filter out non-positive values to avoid log10 issues
    df_filtered = df[df["total_intensity_sum"] > 0].copy()
    df_filtered["log_intensity"] = np.log10(df_filtered["total_intensity_sum"])

    norm_log_intensity = plt.Normalize(
        vmin=df_filtered["log_intensity"].min(),
        vmax=df_filtered["log_intensity"].max()
    )
    cmap_obj = plt.colormaps[cmap]

    fig, ax = plt.subplots(figsize=figsize)

    # Show background image
    ax.imshow(image, cmap='gray', alpha=image_alpha)

    # Plot centroids
    for _, row in df_filtered.iterrows():
        if not np.isnan(row["centroid_x"]) and not np.isnan(row["centroid_y"]):
            color = cmap_obj(norm_log_intensity(row["log_intensity"]))
            circle = patches.Circle(
                (row["centroid_x"], row["centroid_y"]),
                radius=marker_size / 2,
                linewidth=0.5,
                edgecolor="black",
                facecolor=color,
                alpha=0.8
            )
            ax.add_patch(circle)

    # Plot mask contours
    if show_mask_contours and mask is not None:
        contours = measure.find_contours(mask, level=0.5)
        for contour in contours:
            ax.plot(contour[:, 1], contour[:, 0], linewidth=margin_linewidth, color=contour_color)

    # Optional average centroid margin
    if show_margin:
        valid_centroids = df_filtered[["centroid_x", "centroid_y"]].dropna()
        if not valid_centroids.empty:
            avg_x = valid_centroids["centroid_x"].mean()
            avg_y = valid_centroids["centroid_y"].mean()
            ax.plot([avg_x - 100, avg_x + 100], [avg_y - 100, avg_y + 100],
                    color="black", linewidth=margin_linewidth, linestyle="--")

    # Colorbar
    sm = plt.cm.ScalarMappable(cmap=cmap_obj, norm=norm_log_intensity)
    sm.set_array([])
    plt.colorbar(sm, ax=ax, label="log10(Total Intensity Sum)")

    ax.set_aspect('equal')
    ax.set_xlabel("X Coordinate (μm)")
    ax.set_ylabel("Y Coordinate (μm)")
    ax.set_title("m/z Centroids (log10-scaled Intensities)")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [22]:
plot_log_mz_centroids_with_mask_contours(
    df,
    image,
    mask=None,
    cmap="viridis",
    marker_size=1,
    figsize=(6, 4),
    show_margin=False,
    show_mask_contours=True,
    image_alpha=0.3,
    margin_linewidth=1.5,
    contour_color="red"
)

In [23]:

import numpy as np
import pandas as pd
from scipy.stats import entropy
from libpysal.weights import KNN
from esda.moran import Moran
from tqdm import tqdm

def compute_spatial_entropy(intensities):
    intensities = np.array(intensities, dtype=float)
    intensities = intensities[intensities > 0]
    if len(intensities) == 0:
        return 0
    p = intensities / intensities.sum()
    return entropy(p)

def compute_morans_I(x_coords, y_coords, intensities, k=8):
    coords = list(zip(x_coords, y_coords))
    try:
        w = KNN(coords, k=k)
        return Moran(intensities, w).I
    except Exception as e:
        print("Failed with error:", e)
        return np.nan

def analyze_scatter_all(adata, k_neighbors=8):
    num_mz = adata.shape[1]
    mz_values = adata.var["mzs"].values

    x = adata.obs["x"].values
    y = adata.obs["y"].values

    rows = []
    for mz_index in tqdm(range(num_mz), desc="Processing all m/z features"):
        intensities = (
            adata.X[:, mz_index].toarray().flatten()
            if hasattr(adata.X, "toarray")
            else adata.X[:, mz_index].flatten()
        )

        row = {
            "mz": mz_values[mz_index],
            "spatial_entropy": compute_spatial_entropy(intensities),
            "moran_I": compute_morans_I(x, y, intensities, k=k_neighbors),
        }

        rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
"""
Moran's I and Spatial Entropy Analysis
This code computes Moran's I and spatial entropy for each m/z feature in the provided AnnData object.   
Moran's I is a measure of spatial autocorrelation, while spatial entropy quantifies the distribution of intensities. 
Moran's I values close to 1 indicate clustering, while values close to -1 indicate dispersion.
moran's I is computed using the `libpysal` library, which requires spatial coordinates and intensities.
Spatial entropy values range from 0 (uniform distribution) to log(n) (maximal entropy), where n is the number of pixels.   
spatial entropy is computed using the `scipy.stats.entropy` function, which requires the input to be a probability distribution.
The analysis is performed separately for tissue and background regions, and results are stored in a DataFrame.
The DataFrame contains the m/z values, spatial entropy, and Moran's I for both regions.
"""

"\nMoran's I and Spatial Entropy Analysis\nThis code computes Moran's I and spatial entropy for each m/z feature in the provided AnnData object.   \nMoran's I is a measure of spatial autocorrelation, while spatial entropy quantifies the distribution of intensities. \nMoran's I values close to 1 indicate clustering, while values close to -1 indicate dispersion.\nmoran's I is computed using the `libpysal` library, which requires spatial coordinates and intensities.\nSpatial entropy values range from 0 (uniform distribution) to log(n) (maximal entropy), where n is the number of pixels.   \nspatial entropy is computed using the `scipy.stats.entropy` function, which requires the input to be a probability distribution.\nThe analysis is performed separately for tissue and background regions, and results are stored in a DataFrame.\nThe DataFrame contains the m/z values, spatial entropy, and Moran's I for both regions.\n"

In [24]:
# Run it
scatter_wide_df = analyze_scatter_all(adata, k_neighbors=8)

# Save to CSV (optional)
scatter_wide_df.to_csv("all_data_pixel_area_summary.csv", index=False)

# Preview
print(scatter_wide_df.head())

Processing all m/z features: 100%|██████████| 1000/1000 [06:40<00:00,  2.50it/s]

         mz  spatial_entropy   moran_I
0  136.0150         9.261111  0.819859
1  136.0608         9.370870  0.477442
2  137.0251         9.137608  0.854989
3  138.0287         9.191642  0.858436
4  139.0359         9.257034  0.837397


In [25]:
import seaborn as sns
# Plotting Spatial Entropy vs Moran's I for Tissue
plt.figure(figsize=(10, 6))

sns.scatterplot(data=scatter_wide_df, x="spatial_entropy", y="moran_I", hue="mz", palette="viridis")

plt.title("Spatial Entropy vs Moran's I for All m/z")
plt.xlabel("Spatial Entropy")
plt.ylabel("Moran's I")
plt.legend(title="m/z", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()


In [26]:
# Plotting Histograms of Spatial Entropy
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
sns.histplot(scatter_wide_df["spatial_entropy"], bins=30, kde=True, color='blue')
plt.title("Histogram of Spatial Entropy")
plt.xlabel("Spatial Entropy")
plt.ylabel("Frequency")
plt.legend()

# Plotting Histograms of Moran's I
plt.subplot(1, 2, 2)
sns.histplot(scatter_wide_df["moran_I"].dropna(), bins=30, kde=True, color='blue')
plt.title("Histogram of Moran's I")
plt.xlabel("Moran's I")
plt.ylabel("Frequency")
plt.legend()

plt.tight_layout()
plt.show()

/var/folders/0n/w0dydx896p7b5chql0_cv6hr0000gn/T/ipykernel_81766/3761457795.py:9: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend()
/var/folders/0n/w0dydx896p7b5chql0_cv6hr0000gn/T/ipykernel_81766/3761457795.py:17: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend()


In [27]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
plt.scatter(scatter_wide_df['spatial_entropy'], scatter_wide_df['moran_I'], alpha=0.6)
plt.xlabel("Spatial Entropy")
plt.ylabel("Moran's I")
plt.title("Spatial Statistics per m/z")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

/var/folders/0n/w0dydx896p7b5chql0_cv6hr0000gn/T/ipykernel_81766/4005850960.py:8: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend()


In [28]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))
plt.scatter(scatter_wide_df['spatial_entropy'], scatter_wide_df['moran_I'], alpha=0.6)

# Annotate each point with m/z value
for i, row in scatter_wide_df.iterrows():
    mz_label = f"{row['mz']:.1f}"  # format to 1 decimal
    plt.annotate(mz_label,
                 (row['spatial_entropy'], row['moran_I']),
                 textcoords="offset points",
                 xytext=(0, 4),
                 ha='center',
                 fontsize=6,
                 color='blue')

plt.xlabel("Spatial Entropy")
plt.ylabel("Moran's I")
plt.title("Spatial Statistics per m/z")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

/var/folders/0n/w0dydx896p7b5chql0_cv6hr0000gn/T/ipykernel_81766/416552844.py:20: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend()


In [29]:
import numpy as np
import pandas as pd
from tqdm import tqdm

def compute_intensity_metrics_all(adata):
    X = adata.X
    if hasattr(X, "toarray"):  # If sparse matrix
        X = X.toarray()

    mean_intensity = np.mean(X, axis=0)
    total_intensity = np.sum(X, axis=0)

    df = pd.DataFrame({
        "mz": adata.var["mzs"].values,
        "mean_intensity": mean_intensity,
        "total_intensity": total_intensity
    })

    return df

In [30]:
intensity_df = compute_intensity_metrics_all(adata)


In [31]:
results_df = scatter_wide_df.merge(intensity_df, on="mz")


In [32]:
def plot_intensity_vs_moransI_all(df):
    plt.figure(figsize=(10, 6))

    scatter = plt.scatter(
        df["mean_intensity"],
        df["moran_I"],
        c=df["spatial_entropy"],
        s=np.clip(df["total_intensity"] / 1e5, 10, 200),  # adjust scaling if needed
        cmap="viridis",
        alpha=0.8,
        edgecolor='k',
        linewidth=0.5
    )

    plt.xlabel("Mean Intensity")
    plt.ylabel("Moran's I")
    plt.title("m/z Features: Spatial Autocorrelation vs. Intensity")
    cbar = plt.colorbar(scatter)
    cbar.set_label("Spatial Entropy")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [33]:
plot_intensity_vs_moransI_all(results_df)


In [34]:
import matplotlib.pyplot as plt
import numpy as np

def plot_intensity_vs_moransI_with_labels_all(df, top_n=10, score_fn=None):
    plt.figure(figsize=(10, 6))

    # Default score function using combined metrics
    if score_fn is None:
        score_fn = lambda row: row["moran_I"] * np.log1p(row["mean_intensity"]) / (row["spatial_entropy"] + 1e-6)

    df = df.copy()
    df["score"] = df.apply(score_fn, axis=1)

    scatter = plt.scatter(
        df["mean_intensity"],
        df["moran_I"],
        c=df["spatial_entropy"],
        s=np.clip(df["total_intensity"] / 1e5, 10, 200),
        cmap="viridis",
        alpha=0.8,
        edgecolor='k',
        linewidth=0.5
    )

    # Annotate top N points by score
    top_rows = df.nlargest(top_n, "score")
    for _, row in top_rows.iterrows():
        plt.annotate(
            f"{row['mz']:.1f}",
            (row["mean_intensity"], row["moran_I"]),
            fontsize=9,
            xytext=(3, 3),
            textcoords='offset points'
        )

    plt.xlabel("Mean Intensity")
    plt.ylabel("Moran's I")
    plt.title(f"Top {top_n} m/z Features by Spatial-Intensity Score (All Data)")
    cbar = plt.colorbar(scatter)
    cbar.set_label("Spatial Entropy")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [35]:
"""
this function plots the relationship between tissue-to-background intensity ratio and Moran's I for m/z features.
It uses a scatter plot where the size of each point is proportional to the total intensity in tissue, and the color represents spatial entropy.
The function also annotates the top N m/z features based on a custom score, which is calculated as the product of Moran's I and the tissue-to-background ratio, normalized by spatial entropy.
The score is designed to highlight features that are both spatially autocorrelated and have a high intensity ratio.
The function allows for customization of the scoring function, enabling users to define their own criteria for selecting top features.  

"""

"\nthis function plots the relationship between tissue-to-background intensity ratio and Moran's I for m/z features.\nIt uses a scatter plot where the size of each point is proportional to the total intensity in tissue, and the color represents spatial entropy.\nThe function also annotates the top N m/z features based on a custom score, which is calculated as the product of Moran's I and the tissue-to-background ratio, normalized by spatial entropy.\nThe score is designed to highlight features that are both spatially autocorrelated and have a high intensity ratio.\nThe function allows for customization of the scoring function, enabling users to define their own criteria for selecting top features.  \n\n"

In [36]:
plot_intensity_vs_moransI_with_labels_all(results_df, top_n=50)


In [55]:
import matplotlib.pyplot as plt
import numpy as np

def plot_intensity_vs_moransI_with_labels_all(df, top_n=10, score_fn=None):
    plt.figure(figsize=(10, 6))

    # Default score function using combined metrics
    if score_fn is None:
        score_fn = lambda row: row["moran_I"] * np.log1p(row["mean_intensity"]) / (row["spatial_entropy"] + 1e-6)

    df = df.copy()
    df["score"] = df.apply(score_fn, axis=1)

    x_vals = np.log1p(df["mean_intensity"])

    scatter = plt.scatter(
        x_vals,
        df["moran_I"],
        c=df["spatial_entropy"],
        s=np.clip(df["total_intensity"] / 1e5, 10, 200),
        cmap="viridis",
        alpha=0.8,
        edgecolor='k',
        linewidth=0.5
    )

    # Annotate top N points by score
    top_rows = df.nlargest(top_n, "score")
    for _, row in top_rows.iterrows():
        plt.annotate(
            f"{row['mz']:.1f}",
            (np.log1p(row["mean_intensity"]), row["moran_I"]),
            fontsize=9,
            xytext=(3, 3),
            textcoords='offset points'
        )

    plt.xlabel("Log(Mean Intensity + 1)")
    plt.ylabel("Moran's I")
    plt.title(f"Top {top_n} m/z Features by Spatial-Intensity Score (All Data)")
    cbar = plt.colorbar(scatter)
    cbar.set_label("Spatial Entropy")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [56]:
plot_intensity_vs_moransI_with_labels_all(results_df, top_n=50)


In [57]:
results_df

,mz,spatial_entropy,moran_I,mean_intensity,total_intensity
0,136.0150,9.261111,0.819859,443.950075,7371791.0
1,136.0608,9.370870,0.477442,224.911472,3734655.0
2,137.0251,9.137608,0.854989,25784.681903,428154643.0
3,138.0287,9.191642,0.858436,2041.555014,33900021.0
4,139.0359,9.257034,0.837397,451.678410,7500120.0
...,...,...,...,...,...
995,1520.1588,9.043471,0.607464,53.478290,888007.0
996,1521.1633,9.036989,0.601303,47.718880,792372.0
997,1522.1682,9.040476,0.599273,51.754592,859385.0
998,1542.1475,9.075404,0.580004,41.800662,694100.0


In [58]:
import matplotlib.pyplot as plt
import numpy as np

def plot_intensity_vs_moransI_with_labels_all(df, top_n=10, score_fn=None):
    plt.figure(figsize=(10, 6))

    # Default score function using combined metrics (sum intensity now)
    if score_fn is None:
        score_fn = lambda row: row["moran_I"] * np.log1p(row["total_intensity"]) / (row["spatial_entropy"] + 1e-6)

    df = df.copy()
    df["score"] = df.apply(score_fn, axis=1)

    x_vals = np.log1p(df["total_intensity"])

    scatter = plt.scatter(
        x_vals,
        df["moran_I"],
        c=df["spatial_entropy"],
        s=np.clip(df["total_intensity"] / 1e5, 10, 200),
        cmap="viridis",
        alpha=0.8,
        edgecolor='k',
        linewidth=0.5
    )

    # Annotate top N points by score
    top_rows = df.nlargest(top_n, "score")
    for _, row in top_rows.iterrows():
        plt.annotate(
            f"{row['mz']:.1f}",
            (np.log1p(row["total_intensity"]), row["moran_I"]),
            fontsize=9,
            xytext=(3, 3),
            textcoords='offset points'
        )

    plt.xlabel("Log(Total Intensity + 1)")
    plt.ylabel("Moran's I")
    plt.title(f"Top {top_n} m/z Features by Spatial-Intensity Score (All Data)")
    cbar = plt.colorbar(scatter)
    cbar.set_label("Spatial Entropy")
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [59]:
plot_intensity_vs_moransI_with_labels_all(results_df, top_n=50, score_fn=None)


In [64]:
import matplotlib.pyplot as plt
import numpy as np

def plot_entropy_vs_moransI_with_labels_all(df, top_n=10, score_fn=None):
    plt.figure(figsize=(10, 6))

    # Default score function using spatial entropy and total intensity
    if score_fn is None:
        score_fn = lambda row: row["moran_I"] * np.log1p(row["spatial_entropy"]) / (row["total_intensity"] + 1e-6)

    df = df.copy()
    df["score"] = df.apply(score_fn, axis=1)

    x_vals = df["spatial_entropy"]

    scatter = plt.scatter(
        x_vals,
        df["moran_I"],
        c=np.log1p(df["total_intensity"]),
        s=np.clip(df["total_intensity"] / 1e5, 10, 200),
        cmap="viridis",
        alpha=0.8,
        edgecolor='k',
        linewidth=0.5
    )

    # Annotate top N points by score
    top_rows = df.nlargest(top_n, "score")
    for _, row in top_rows.iterrows():
        plt.annotate(
            f"{row['mz']:.1f}",
            (row["spatial_entropy"], row["moran_I"]),
            fontsize=9,
            xytext=(3, 3),
            textcoords='offset points'
        )

    plt.xlabel("Spatial Entropy")
    plt.ylabel("Moran's I")
    plt.title(f"Top {top_n} m/z Features by Entropy-Moran Score (All Data)")
    cbar = plt.colorbar(scatter)
    cbar.set_label("Log(Total Intensity + 1)")
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [65]:
plot_entropy_vs_moransI_with_labels_all(results_df, top_n=50, score_fn=None)


In [ ]:

clustered_df = results_df[results_df["moran_I"] > 0.5]
clustered_df = clustered_df[clustered_df["spatial_entropy"] < 1.5]
# Ensure the image folder exists
os.makedirs(f"images_{foldchange}", exist_ok=True)

# Compute the global spectrum
global_spectrum = adata.X.sum(axis=0).A1 if hasattr(adata.X, "A1") else np.asarray(adata.X.sum(axis=0)).flatten()

# Get indices of m/z values sorted by intensity (descending)
sorted_indices = np.argsort(global_spectrum)[::-1]

# Select indices from 10th to 20th (ranked #10 to #20)
selected_indices = sorted_indices[0:]

# Get corresponding m/z values
selected_mz = adata.var_names[selected_indices].astype(float)
"""
# Modified plotting loop
for mz in selected_mz:
    # Generate figure using your existing function
    fig = plot_mz_across_sample_custom_tic(adata, mz, tol=0.001)  # Modify function to return fig
    
    # Define filename
    filename = f"images_{foldchange}/mz_{mz:.4f}.png"
    
    # Save the figure
    pio.write_image(fig, filename)
    print(f"Saved: {filename}")

"""